In [3]:
import math
import re
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from datetime import date, timedelta
from IPython.display import display, clear_output, HTML, FileLink

# Enable interactive widgets in Google Colab
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass


# ============================================================
# Default values shown when the form first opens
# ============================================================

DEFAULT_COMPLETED = {
    date(2026, 9, 5): 8,   # PB-03
    date(2026, 9, 7): 8,   # PB-04
}


# ============================================================
# Main form controls
# ============================================================

common_style = {
    "description_width": "170px"
}

title_input = widgets.Text(
    value="Sprint 1 Burn-down Chart",
    description="Chart title:",
    style=common_style,
    layout=widgets.Layout(width="650px")
)

total_sp_input = widgets.BoundedIntText(
    value=56,
    min=1,
    max=10000,
    description="Initial Story Points:",
    style=common_style,
    layout=widgets.Layout(width="350px")
)

start_date_input = widgets.DatePicker(
    value=date(2026, 8, 31),
    description="Sprint start date:",
    style=common_style,
    layout=widgets.Layout(width="350px")
)

end_date_input = widgets.DatePicker(
    value=date(2026, 9, 18),
    description="Sprint end date:",
    style=common_style,
    layout=widgets.Layout(width="350px")
)

current_date_input = widgets.DatePicker(
    value=date(2026, 9, 7),
    description="Data current as of:",
    style=common_style,
    layout=widgets.Layout(width="350px")
)

filename_input = widgets.Text(
    value="S1_Burn_Down_Chart",
    description="Output filename:",
    style=common_style,
    layout=widgets.Layout(width="500px")
)

line_style_input = widgets.Dropdown(
    options=[
        ("Normal line", "line"),
        ("Step line", "step"),
    ],
    value="line",
    description="Actual line style:",
    style=common_style,
    layout=widgets.Layout(width="350px")
)

show_values_input = widgets.Checkbox(
    value=False,
    description="Show remaining SP above each point",
    indent=False,
    layout=widgets.Layout(width="350px")
)


# ============================================================
# Buttons and output areas
# ============================================================

refresh_button = widgets.Button(
    description="Refresh daily input table",
    button_style="info",
    icon="refresh",
    layout=widgets.Layout(width="250px")
)

generate_button = widgets.Button(
    description="Generate chart",
    button_style="success",
    icon="line-chart",
    layout=widgets.Layout(width="200px")
)

download_png_button = widgets.Button(
    description="Download PNG",
    button_style="primary",
    icon="download",
    disabled=True,
    layout=widgets.Layout(width="180px")
)

download_pdf_button = widgets.Button(
    description="Download PDF",
    button_style="primary",
    icon="download",
    disabled=True,
    layout=widgets.Layout(width="180px")
)

daily_input_area = widgets.Output()
chart_output_area = widgets.Output()
message_output_area = widgets.Output()

daily_completed_inputs = {}
generated_files = {}


# ============================================================
# Create editable daily Story Point fields
# ============================================================

def refresh_daily_inputs(_=None):
    previous_values = {
        day_date: field.value
        for day_date, field in daily_completed_inputs.items()
    }

    daily_completed_inputs.clear()

    with daily_input_area:
        clear_output(wait=True)

        sprint_start = start_date_input.value
        sprint_end = end_date_input.value
        current_date = current_date_input.value

        if not sprint_start or not sprint_end or not current_date:
            display(HTML(
                "<p style='color:red;'>Please select all three dates.</p>"
            ))
            return False

        if sprint_end < sprint_start:
            display(HTML(
                "<p style='color:red;'>"
                "Sprint end date cannot be earlier than the start date."
                "</p>"
            ))
            return False

        if current_date < sprint_start:
            display(HTML(
                "<p style='color:red;'>"
                "Current date cannot be earlier than the Sprint start date."
                "</p>"
            ))
            return False

        if current_date > sprint_end:
            display(HTML(
                "<p style='color:red;'>"
                "Current date cannot be later than the Sprint end date."
                "</p>"
            ))
            return False

        number_of_days = (current_date - sprint_start).days + 1

        header = widgets.HBox([
            widgets.HTML(
                "<b>Day and date</b>",
                layout=widgets.Layout(width="260px")
            ),
            widgets.HTML(
                "<b>Completed Story Points</b>",
                layout=widgets.Layout(width="220px")
            ),
        ])

        rows = [header]

        for index in range(number_of_days):
            day_number = index + 1
            day_date = sprint_start + timedelta(days=index)

            default_value = previous_values.get(
                day_date,
                DEFAULT_COMPLETED.get(day_date, 0)
            )

            completed_field = widgets.BoundedIntText(
                value=default_value,
                min=0,
                max=10000,
                step=1,
                layout=widgets.Layout(width="150px")
            )

            daily_completed_inputs[day_date] = completed_field

            date_label = widgets.HTML(
                value=(
                    f"<b>Day {day_number}</b> — "
                    f"{day_date.strftime('%d %B %Y')}"
                ),
                layout=widgets.Layout(width="260px")
            )

            rows.append(
                widgets.HBox([
                    date_label,
                    completed_field
                ])
            )

        display(widgets.VBox(rows))

    return True


# ============================================================
# Generate burn-down chart
# ============================================================

def generate_chart(_=None):
    with message_output_area:
        clear_output(wait=True)

    if not refresh_daily_inputs():
        return

    sprint_start = start_date_input.value
    sprint_end = end_date_input.value
    current_date = current_date_input.value

    initial_sp = total_sp_input.value
    chart_title = title_input.value.strip()

    total_sprint_days = (sprint_end - sprint_start).days + 1
    current_day_number = (current_date - sprint_start).days + 1

    actual_days = [0]
    actual_remaining = [initial_sp]

    remaining_sp = initial_sp
    daily_results = []

    for day_number in range(1, current_day_number + 1):
        day_date = sprint_start + timedelta(days=day_number - 1)
        completed_sp = daily_completed_inputs[day_date].value

        remaining_sp -= completed_sp

        if remaining_sp < 0:
            with message_output_area:
                display(HTML(
                    "<p style='color:red;'>"
                    "The total completed Story Points exceed the initial "
                    "committed Story Points. Please check the daily values."
                    "</p>"
                ))
            return

        actual_days.append(day_number)
        actual_remaining.append(remaining_sp)

        daily_results.append({
            "day": day_number,
            "date": day_date,
            "completed": completed_sp,
            "remaining": remaining_sp
        })

    ideal_days = np.arange(total_sprint_days + 1)

    ideal_remaining = np.linspace(
        initial_sp,
        0,
        total_sprint_days + 1
    )

    day_labels = ["Day 0\nStart"]

    for day_number in range(1, total_sprint_days + 1):
        day_date = sprint_start + timedelta(days=day_number - 1)

        day_labels.append(
            f"Day {day_number}\n{day_date.strftime('%d %b')}"
        )

    fig, ax = plt.subplots(figsize=(15, 9))

    if line_style_input.value == "step":
        ax.step(
            actual_days,
            actual_remaining,
            where="post",
            color="blue",
            linewidth=2.5,
            label="Actual Remaining Effort"
        )

        ax.scatter(
            actual_days,
            actual_remaining,
            color="blue",
            s=60,
            zorder=3
        )

    else:
        ax.plot(
            actual_days,
            actual_remaining,
            color="blue",
            marker="o",
            linewidth=2.5,
            markersize=7,
            label="Actual Remaining Effort"
        )

    ax.plot(
        ideal_days,
        ideal_remaining,
        color="red",
        linestyle="--",
        linewidth=2.5,
        label="Ideal Burn Down"
    )

    if show_values_input.value:
        for day_number, remaining in zip(
            actual_days,
            actual_remaining
        ):
            ax.annotate(
                str(remaining),
                (day_number, remaining),
                textcoords="offset points",
                xytext=(0, 10),
                ha="center",
                fontsize=10,
                color="blue"
            )

    ax.set_title(
        chart_title,
        fontsize=24,
        fontweight="bold",
        pad=20
    )

    ax.set_xlabel(
        "Sprint Calendar Day",
        fontsize=14
    )

    ax.set_ylabel(
        "Remaining Effort (Story Points)",
        fontsize=14
    )

    ax.set_xticks(ideal_days)

    ax.set_xticklabels(
        day_labels,
        rotation=45,
        ha="right",
        fontsize=10
    )

    ax.set_xlim(0, total_sprint_days)

    highest_value = max(
        initial_sp,
        max(actual_remaining)
    )

    y_axis_max = max(
        10,
        math.ceil((highest_value * 1.07) / 5) * 5
    )

    ax.set_ylim(0, y_axis_max)

    ax.grid(
        True,
        linestyle="--",
        alpha=0.35
    )

    ax.legend(
        loc="upper right",
        fontsize=12
    )

    ax.text(
        0.985,
        0.02,
        f"Data current as of {current_date.strftime('%Y-%m-%d')}",
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=10,
        color="gray"
    )

    plt.tight_layout()

    safe_filename = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        filename_input.value.strip()
    )

    if not safe_filename:
        safe_filename = "Burn_Down_Chart"

    png_filename = f"{safe_filename}.png"
    pdf_filename = f"{safe_filename}.pdf"

    fig.savefig(
        png_filename,
        dpi=300,
        bbox_inches="tight"
    )

    fig.savefig(
        pdf_filename,
        dpi=300,
        bbox_inches="tight"
    )

    generated_files["png"] = png_filename
    generated_files["pdf"] = pdf_filename

    download_png_button.disabled = False
    download_pdf_button.disabled = False

    with chart_output_area:
        clear_output(wait=True)
        display(fig)
        plt.close(fig)

        display(HTML(
            f"""
            <p>
                <b>Initial Story Points:</b> {initial_sp}<br>
                <b>Remaining Story Points:</b> {remaining_sp}<br>
                <b>Data current as of:</b>
                {current_date.strftime('%d %B %Y')}
            </p>
            """
        ))

        table_rows = ""

        for row in daily_results:
            table_rows += f"""
            <tr>
                <td style="padding:6px;border:1px solid #ccc;">
                    Day {row['day']}
                </td>
                <td style="padding:6px;border:1px solid #ccc;">
                    {row['date'].strftime('%d %B %Y')}
                </td>
                <td style="padding:6px;border:1px solid #ccc;">
                    {row['completed']}
                </td>
                <td style="padding:6px;border:1px solid #ccc;">
                    {row['remaining']}
                </td>
            </tr>
            """

        display(HTML(
            f"""
            <table style="border-collapse:collapse;">
                <tr>
                    <th style="padding:6px;border:1px solid #ccc;">Day</th>
                    <th style="padding:6px;border:1px solid #ccc;">Date</th>
                    <th style="padding:6px;border:1px solid #ccc;">
                        Completed SP
                    </th>
                    <th style="padding:6px;border:1px solid #ccc;">
                        Remaining SP
                    </th>
                </tr>
                {table_rows}
            </table>
            """
        ))

    with message_output_area:
        display(HTML(
            f"""
            <p style="color:green;">
                Chart generated successfully:<br>
                <b>{png_filename}</b><br>
                <b>{pdf_filename}</b>
            </p>
            """
        ))


# ============================================================
# Download buttons
# ============================================================

def download_png(_=None):
    if "png" not in generated_files:
        return

    try:
        from google.colab import files
        files.download(generated_files["png"])
    except ImportError:
        display(FileLink(generated_files["png"]))


def download_pdf(_=None):
    if "pdf" not in generated_files:
        return

    try:
        from google.colab import files
        files.download(generated_files["pdf"])
    except ImportError:
        display(FileLink(generated_files["pdf"]))


refresh_button.on_click(refresh_daily_inputs)
generate_button.on_click(generate_chart)
download_png_button.on_click(download_png)
download_pdf_button.on_click(download_pdf)


# ============================================================
# Display the editable interface
# ============================================================

display(HTML("<h2>Sprint Burn-down Chart Generator</h2>"))

display(HTML(
    """
    <p>
        Edit the fields below, refresh the daily input table,
        enter the Story Points completed on each day, and then
        generate the chart.
    </p>
    """
))

display(widgets.VBox([
    title_input,
    total_sp_input,
    start_date_input,
    end_date_input,
    current_date_input,
    filename_input,
    line_style_input,
    show_values_input
]))

display(HTML("<h3>Daily completed Story Points</h3>"))
display(refresh_button)
display(daily_input_area)

display(HTML("<br>"))
display(generate_button)

display(widgets.HBox([
    download_png_button,
    download_pdf_button
]))

display(message_output_area)
display(chart_output_area)

refresh_daily_inputs()

Button(button_style='info', description='Refresh daily input table', icon='refresh', layout=Layout(width='250p…

Output()

Button(button_style='success', description='Generate chart', icon='line-chart', layout=Layout(width='200px'), …

Output()

Output()

True

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>